# Meridian FairLend AI - Model Training & Explainability (XAI)
### Mission Bharat • Retail Credit Scoring Engine
This notebook trains and evaluates the Explainable Loan Approval Model using **Logistic Regression**, **Random Forest**, and **XGBoost**, with **SHAP** (SHapley Additive exPlanations) for regulatory transparency.

In [ ]:
# Step 1: Install & Import Libraries
!pip install -q scikit-learn pandas numpy matplotlib seaborn shap xgboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import shap
import pickle

print('All libraries successfully loaded!')

### Step 2: Load & Explore the Dataset

In [ ]:
# If uploading to Google Colab, upload 'train_u6lujuX_CVtuZ9i (1).csv'
# or use the direct raw URL:
url = 'https://raw.githubusercontent.com/datasets/loan-dataset/master/train.csv'
try:
    df = pd.read_csv('train_u6lujuX_CVtuZ9i (1).csv')
except Exception:
    try:
        df = pd.read_csv('E:/archive/train_u6lujuX_CVtuZ9i (1).csv')
    except Exception:
        df = pd.read_csv('https://raw.githubusercontent.com/dphi-official/Datasets/master/Loan_Data/loan_train.csv')

print(f'Dataset Shape: {df.shape}')
df.head()

### Step 3: Feature Engineering & Preprocessing
We calculate:
1. **Total Income** = ApplicantIncome + CoapplicantIncome
2. **Monthly EMI** = (LoanAmount * 1000) / Loan_Amount_Term
3. **DTI Ratio** = (Monthly EMI / Total Income) * 100

In [ ]:
# Handle Missing Values
df['Gender'].fillna(df['Gender'].mode()[0], inplace=True)
df['Married'].fillna(df['Married'].mode()[0], inplace=True)
df['Dependents'].fillna(df['Dependents'].mode()[0], inplace=True)
df['Self_Employed'].fillna(df['Self_Employed'].mode()[0], inplace=True)
df['LoanAmount'].fillna(df['LoanAmount'].median(), inplace=True)
df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0], inplace=True)
df['Credit_History'].fillna(1.0, inplace=True)

# Feature Engineering
df['Total_Income'] = df['ApplicantIncome'] + df['CoapplicantIncome']
df['Monthly_EMI'] = (df['LoanAmount'] * 1000) / df['Loan_Amount_Term']
df['DTI_Percent'] = (df['Monthly_EMI'] / df['Total_Income'].clip(lower=1)) * 100

# Target variable mapping
target_col = 'Loan_Status' if 'Loan_Status' in df.columns else 'status'
y = df[target_col].map({'Y': 1, 'N': 0, 1: 1, 0: 0})

# One-Hot Encoding
features = ['Credit_History', 'DTI_Percent', 'Total_Income', 'LoanAmount', 'Loan_Amount_Term', 'Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area']
X = pd.get_dummies(df[features], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Training shape: {X_train.shape}, Test shape: {X_test.shape}')

### Step 4: Model Training (Logistic Regression - Transparent L2 Baseline)

In [ ]:
model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f'Accuracy:  {accuracy_score(y_test, y_pred) * 100:.2f}%')
print(f'Precision: {precision_score(y_test, y_pred) * 100:.2f}%')
print(f'Recall:    {recall_score(y_test, y_pred) * 100:.2f}%')
print(f'F1-Score:  {f1_score(y_test, y_pred) * 100:.2f}%')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob):.3f}')

### Step 5: SHAP Explainability (Feature Attribution)

In [ ]:
explainer = shap.LinearExplainer(model, X_train, feature_perturbation='interventional')
shap_values = explainer.shap_values(X_test)

# Summary plot showing global factor importance
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('Meridian FairLend AI - Global Feature Importance (SHAP)')
plt.tight_layout()
plt.show()

### Step 6: Export Trained Model for Meridian FairLend

In [ ]:
with open('meridian_loan_model.pkl', 'wb') as f:
    pickle.dump({'model': model, 'feature_names': list(X.columns)}, f)
print('Model saved successfully to meridian_loan_model.pkl!')